# SSVEP Pipeline: MATLAB Reproduction

This notebook consolidates the entire `ssvep_pipeline` package (`config.py`, `mat_io.py`, `features.py`, `evaluation.py`), the `run_pipeline.py` driver script, and `validate_against_matlab.py` into a single runnable notebook.

**Paths used below** (adjust if your layout differs):
- Dataset directory: `../data/benchmark` (contains `S1.mat` ... `S35.mat`, `Freq_Phase.mat`, `64-channels.loc`)
- Output directory: `../results` (all CSVs, figures, and fitted models are written here)

**Sections**
1. Imports
2. Configuration (`config.py`)
3. MATLAB I/O helpers (`mat_io.py`)
4. Feature extraction (`features.py`)
5. Evaluation (`evaluation.py`)
6. Pipeline driver (`run_pipeline.py`)
7. Run the pipeline
8. Validation against MATLAB (`validate_against_matlab.py`)

In [30]:
from __future__ import annotations

import sys
from pathlib import Path

import h5py
import joblib
import numpy as np
import pandas as pd
import pywt

from itertools import permutations
from dataclasses import dataclass, field
from typing import Any, Callable, Iterable

from scipy.io import loadmat

from sklearn.base import clone
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import (
    StratifiedKFold,
    StratifiedShuffleSplit,
    cross_val_predict,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

pd.set_option("display.max_columns", 50)


## 2. Configuration (`config.py`)

Defines the frozen `PipelineConfig` dataclass holding all pipeline parameters: sampling rate, wavelet settings, FFT band, subject groupings, and cross-validation settings.

In [31]:
@dataclass(frozen=True)
class PipelineConfig:
    dataset_dir: Path
    output_dir: Path

    sampling_frequency_hz: float = 250.0
    full_trial_samples: int = 1500
    oz_channel_matlab_index: int = 62

    stimulus_start_matlab_index: int = 126
    stimulus_end_matlab_index: int = 1375

    target_frequencies_hz: tuple[float, ...] = (9.0, 10.0, 11.0, 12.0)

    wavelet_name: str = "db4"
    wavelet_level: int = 4
    wavelet_mode: str = "symmetric"

    fft_lower_hz: float = 7.8
    fft_upper_hz: float = 15.6

    group_a_subjects: tuple[int, ...] = (
        10, 14, 15, 20, 22, 25, 26, 31, 32, 35
    )

    group_b_subjects: tuple[int, ...] = (
        1, 2, 3, 5, 6, 7, 8, 9, 12,
        17, 18, 19, 21, 24, 27, 28, 30, 34
    )

    blocks_per_class: int = 6
    expected_channels: int = 64
    expected_stimuli: int = 40

    random_seed: int = 2026
    holdout_repetitions: int = 100
    holdout_fraction: float = 0.20

    save_intermediate_d4: bool = False

    @property
    def oz_channel_python_index(self) -> int:
        return self.oz_channel_matlab_index - 1

    @property
    def stimulus_slice(self) -> slice:
        start = self.stimulus_start_matlab_index - 1
        stop = self.stimulus_end_matlab_index
        return slice(start, stop)

    @property
    def stimulus_samples(self) -> int:
        return (
            self.stimulus_end_matlab_index
            - self.stimulus_start_matlab_index
            + 1
        )

    def subjects_for_group(self, group_name: str) -> tuple[int, ...]:
        normalized = group_name.strip().upper()
        if normalized == "A":
            return self.group_a_subjects
        if normalized == "B":
            return self.group_b_subjects
        raise ValueError("group_name must be \'A\' or \'B\'.")

    def validate(self) -> None:
        if not self.dataset_dir.is_dir():
            raise FileNotFoundError(
                f"Dataset directory does not exist: {self.dataset_dir}"
            )

        if self.stimulus_samples != 1250:
            raise ValueError(
                f"Expected a 1250-sample stimulus window, got "
                f"{self.stimulus_samples}."
            )

        if self.oz_channel_python_index != 61:
            raise ValueError(
                "Oz must be MATLAB channel 62 / Python channel index 61."
            )

        if not 0.0 < self.holdout_fraction < 1.0:
            raise ValueError("holdout_fraction must be between 0 and 1.")
        if self.holdout_repetitions < 1:
            raise ValueError("holdout_repetitions must be at least 1.")


## 3. MATLAB I/O helpers (`mat_io.py`)

Locates subject/frequency `.mat` files, loads them (supporting both MATLAB v7 and v7.3/HDF5 formats), normalizes the EEG array to the expected `(64 channels, 1500 samples, 40 stimuli, 6 blocks)` shape, and maps target stimulus frequencies to dataset indices.

In [32]:
EXPECTED_EEG_SHAPE = (64, 1500, 40, 6)


def _find_unique_file(root: Path, candidate_names: tuple[str, ...]) -> Path:
    matches: list[Path] = []
    for name in candidate_names:
        matches.extend(root.rglob(name))

    unique_matches = sorted(set(path.resolve() for path in matches))

    if not unique_matches:
        raise FileNotFoundError(
            f"None of these files were found under {root}: "
            f"\', \'.join(candidate_names)"
        )

    if len(unique_matches) > 1:
        exact_name_matches = [
            path for path in unique_matches if path.name in candidate_names
        ]
        if len(exact_name_matches) == 1:
            return exact_name_matches[0]
        raise RuntimeError(
            "Multiple candidate files were found:\n"
            + "\n".join(str(path) for path in unique_matches)
        )

    return unique_matches[0]


def find_subject_file(dataset_dir: Path, subject_id: int) -> Path:
    return _find_unique_file(
        dataset_dir,
        (f"S{subject_id}.mat", f"S{subject_id:02d}.mat"),
    )


def find_frequency_file(dataset_dir: Path) -> Path:
    return _find_unique_file(dataset_dir, ("Freq_Phase.mat",))


def _load_mat_v7(path: Path) -> dict[str, Any]:
    loaded = loadmat(path, squeeze_me=False, struct_as_record=False)
    return {
        key: value
        for key, value in loaded.items()
        if not key.startswith("__")
    }


def _load_mat_v73(path: Path) -> dict[str, np.ndarray]:
    result: dict[str, np.ndarray] = {}
    with h5py.File(path, "r") as handle:
        for key, value in handle.items():
            if isinstance(value, h5py.Dataset):
                result[key] = np.array(value)
    return result


def load_mat_any(path: Path) -> dict[str, Any]:
    try:
        return _load_mat_v7(path)
    except (NotImplementedError, ValueError, OSError):
        return _load_mat_v73(path)


def normalize_eeg_shape(array: np.ndarray) -> np.ndarray:
    eeg = np.asarray(array)

    if eeg.ndim != 4:
        raise ValueError(
            f"Expected a 4-D EEG array, received shape {eeg.shape}."
        )

    if eeg.shape == EXPECTED_EEG_SHAPE:
        return np.asarray(eeg, dtype=np.float64)

    matching_permutations: list[tuple[int, ...]] = []
    for order in permutations(range(4)):
        if tuple(eeg.shape[index] for index in order) == EXPECTED_EEG_SHAPE:
            matching_permutations.append(order)

    if len(matching_permutations) != 1:
        raise ValueError(
            f"Could not uniquely transform EEG shape {eeg.shape} into "
            f"{EXPECTED_EEG_SHAPE}. Matching permutations: "
            f"{matching_permutations}"
        )

    normalized = np.transpose(eeg, matching_permutations[0])
    return np.asarray(normalized, dtype=np.float64)


def load_subject_eeg(path: Path) -> np.ndarray:
    loaded = load_mat_any(path)

    if "data" in loaded:
        return normalize_eeg_shape(np.asarray(loaded["data"]))

    candidates: list[np.ndarray] = []
    for value in loaded.values():
        if isinstance(value, np.ndarray) and value.ndim == 4:
            try:
                candidates.append(normalize_eeg_shape(value))
            except ValueError:
                continue

    if len(candidates) != 1:
        raise ValueError(
            f"Expected one EEG array in {path}, found {len(candidates)}."
        )

    return candidates[0]


def load_dataset_frequencies(path: Path) -> np.ndarray:
    loaded = load_mat_any(path)

    if "freqs" not in loaded:
        raise KeyError(f"Variable \'freqs\' was not found in {path}.")

    frequencies = np.asarray(loaded["freqs"], dtype=np.float64).reshape(-1)

    if frequencies.size != 40:
        raise ValueError(
            f"Expected 40 stimulus frequencies, found {frequencies.size}."
        )

    return frequencies


def map_target_frequency_indices(
    dataset_frequencies: np.ndarray,
    targets_hz: tuple[float, ...],
) -> dict[float, int]:
    mapping: dict[float, int] = {}

    for target in targets_hz:
        matches = np.flatnonzero(
            np.isclose(dataset_frequencies, target, atol=1e-10, rtol=0.0)
        )
        if matches.size != 1:
            raise ValueError(
                f"Expected exactly one dataset index for {target} Hz, "
                f"found {matches.size}."
            )
        mapping[target] = int(matches[0])

    return mapping


## 4. Feature extraction (`features.py`)

For each trial, reconstructs the D4 wavelet detail sub-band from the Oz channel, crops to the stimulus window, takes the FFT, and finds the dominant frequency inside the SSVEP band. Results are aggregated into `all_trial_features.csv` plus per-group Classification-Learner-ready files.

In [33]:
def reconstruct_d4_full_trial(
    full_oz_trial: np.ndarray,
    config: PipelineConfig,
) -> np.ndarray:
    signal = np.asarray(full_oz_trial, dtype=np.float64).reshape(-1)

    if signal.size != config.full_trial_samples:
        raise ValueError(
            f"Expected {config.full_trial_samples} samples, got "
            f"{signal.size}."
        )

    coefficients = pywt.wavedec(
        signal,
        wavelet=config.wavelet_name,
        mode=config.wavelet_mode,
        level=config.wavelet_level,
    )

    if len(coefficients) != config.wavelet_level + 1:
        raise RuntimeError(
            f"Unexpected number of wavelet coefficient arrays: "
            f"{len(coefficients)}."
        )

    isolated = [np.zeros_like(coefficient) for coefficient in coefficients]

    d4_position = 1
    isolated[d4_position] = coefficients[d4_position].copy()

    reconstructed = pywt.waverec(
        isolated,
        wavelet=config.wavelet_name,
        mode=config.wavelet_mode,
    )

    reconstructed = np.asarray(reconstructed, dtype=np.float64).reshape(-1)
    reconstructed = reconstructed[: config.full_trial_samples]

    if reconstructed.size != config.full_trial_samples:
        raise RuntimeError(
            f"Reconstructed D4 has {reconstructed.size} samples; expected "
            f"{config.full_trial_samples}."
        )

    return reconstructed


def dominant_frequency_from_d4(
    full_trial_d4: np.ndarray,
    config: PipelineConfig,
) -> tuple[float, float]:
    stimulus_d4 = np.asarray(
        full_trial_d4[config.stimulus_slice],
        dtype=np.float64,
    )

    if stimulus_d4.size != config.stimulus_samples:
        raise RuntimeError(
            f"Expected {config.stimulus_samples} cropped D4 samples, got "
            f"{stimulus_d4.size}."
        )

    n_samples = stimulus_d4.size
    fft_values = np.fft.rfft(stimulus_d4, n=n_samples)
    frequencies = np.fft.rfftfreq(
        n_samples,
        d=1.0 / config.sampling_frequency_hz,
    )

    magnitude = np.abs(fft_values) / n_samples
    if magnitude.size > 2:
        magnitude[1:-1] *= 2.0

    band_mask = (
        (frequencies >= config.fft_lower_hz - 1e-12)
        & (frequencies <= config.fft_upper_hz + 1e-12)
    )

    if not np.any(band_mask):
        raise RuntimeError("No FFT bins fall inside the configured band.")

    band_frequencies = frequencies[band_mask]
    band_magnitudes = magnitude[band_mask]

    maximum_index = int(np.argmax(band_magnitudes))

    return (
        float(band_frequencies[maximum_index]),
        float(band_magnitudes[maximum_index]),
    )


def extract_trial_feature(
    full_oz_trial: np.ndarray,
    config: PipelineConfig,
) -> tuple[float, float, np.ndarray]:
    full_trial_d4 = reconstruct_d4_full_trial(full_oz_trial, config)
    dominant_hz, dominant_amplitude = dominant_frequency_from_d4(
        full_trial_d4,
        config,
    )
    return dominant_hz, dominant_amplitude, full_trial_d4


def extract_features(
    config: PipelineConfig,
    groups: Iterable[str] = ("A", "B"),
) -> pd.DataFrame:
    config.validate()
    config.output_dir.mkdir(parents=True, exist_ok=True)

    frequency_file = find_frequency_file(config.dataset_dir)
    dataset_frequencies = load_dataset_frequencies(frequency_file)
    frequency_indices = map_target_frequency_indices(
        dataset_frequencies,
        config.target_frequencies_hz,
    )

    print("Verified target-frequency mapping:")
    for class_label, target_hz in enumerate(
        config.target_frequencies_hz,
        start=1,
    ):
        python_index = frequency_indices[target_hz]
        print(
            f"Class {class_label}: {target_hz:.1f} Hz -> "
            f"dataset index {python_index + 1} (MATLAB)"
        )
    print()

    rows: list[dict[str, float | int | str]] = []
    intermediate_d4: list[np.ndarray] = []

    for group_name in groups:
        normalized_group = group_name.strip().upper()
        subjects = config.subjects_for_group(normalized_group)

        for subject_position, subject_id in enumerate(subjects, start=1):
            subject_file = find_subject_file(
                config.dataset_dir,
                subject_id,
            )

            eeg = load_subject_eeg(subject_file)

            if not np.isfinite(eeg).all():
                raise ValueError(
                    f"NaN or Inf values were found in S{subject_id}."
                )

            print(
                f"Group {normalized_group}: processing S{subject_id} "
                f"({subject_position}/{len(subjects)})"
            )

            for class_label, target_hz in enumerate(
                config.target_frequencies_hz,
                start=1,
            ):
                stimulus_index = frequency_indices[target_hz]

                for block_python_index in range(config.blocks_per_class):
                    full_oz_trial = eeg[
                        config.oz_channel_python_index,
                        :,
                        stimulus_index,
                        block_python_index,
                    ]

                    dominant_hz, dominant_amplitude, full_trial_d4 = (
                        extract_trial_feature(full_oz_trial, config)
                    )

                    rows.append({
                        "Group": normalized_group,
                        "SubjectID": subject_id,
                        "ClassLabel": class_label,
                        "TargetFrequencyHz": target_hz,
                        "Block": block_python_index + 1,
                        "DominantFrequencyHz": dominant_hz,
                        "DominantAmplitude": dominant_amplitude,
                    })

                    if config.save_intermediate_d4:
                        intermediate_d4.append(full_trial_d4)

    features = pd.DataFrame(rows)

    expected_rows = (
        sum(len(config.subjects_for_group(group)) for group in groups)
        * len(config.target_frequencies_hz)
        * config.blocks_per_class
    )

    if len(features) != expected_rows:
        raise RuntimeError(
            f"Expected {expected_rows} feature rows, got {len(features)}."
        )

    features_file = config.output_dir / "all_trial_features.csv"
    features.to_csv(features_file, index=False)

    for group_name in sorted(features["Group"].unique()):
        group_features = features.loc[
            features["Group"] == group_name,
            ["DominantFrequencyHz", "ClassLabel"],
        ].reset_index(drop=True)

        group_features.to_csv(
            config.output_dir
            / f"Group{group_name}_ClassificationLearner.csv",
            index=False,
        )

        group_features.to_excel(
            config.output_dir
            / f"Group{group_name}_ClassificationLearner.xlsx",
            index=False,
        )

    subject_class_means = (
        features.groupby(
            ["Group", "SubjectID", "ClassLabel", "TargetFrequencyHz"],
            as_index=False,
        )["DominantFrequencyHz"]
        .mean()
        .rename(columns={"DominantFrequencyHz": "MeanDominantFrequencyHz"})
    )

    subject_class_means.to_csv(
        config.output_dir / "subject_class_feature_means.csv",
        index=False,
    )

    if config.save_intermediate_d4:
        d4_array = np.stack(intermediate_d4, axis=0)
        np.save(
            config.output_dir / "full_trial_reconstructed_d4.npy",
            d4_array,
        )

    return features


## 5. Evaluation (`evaluation.py`)

Builds three classifiers (Fine Tree, Linear Discriminant, Linear SVM), runs stratified 10-fold CV and 100x repeated holdout evaluation, compares against the paper's reported accuracies, and saves fitted models plus summary CSVs.

In [34]:
PAPER_RESULTS = {
    "A": {
        "ten_fold_accuracy_percent": {
            "Fine Tree": 95.8,
            "Linear Discriminant": 96.7,
            "Linear SVM": 96.7,
        },
        "holdout_mean_percent": {
            "Fine Tree": 96.74,
            "Linear Discriminant": 97.66,
            "Linear SVM": 97.66,
        },
        "holdout_sd_percent": {
            "Fine Tree": 2.67,
            "Linear Discriminant": 2.35,
            "Linear SVM": 2.35,
        },
    },
    "B": {
        "ten_fold_accuracy_percent": {
            "Fine Tree": 91.4,
            "Linear Discriminant": 91.7,
            "Linear SVM": 91.9,
        },
        "holdout_mean_percent": {
            "Fine Tree": 90.46,
            "Linear Discriminant": 91.60,
            "Linear SVM": 92.68,
        },
        "holdout_sd_percent": {
            "Fine Tree": 2.22,
            "Linear Discriminant": 0.82,
            "Linear SVM": 0.67,
        },
    },
}


def build_models(seed: int) -> dict[str, object]:
    return {
        "Fine Tree": DecisionTreeClassifier(
            criterion="gini",
            max_leaf_nodes=101,
            random_state=seed,
        ),
        "Linear Discriminant": LinearDiscriminantAnalysis(
            solver="svd",
        ),
        "Linear SVM": Pipeline(
            steps=[
                ("standardize", StandardScaler()),
                (
                    "svm",
                    SVC(
                        kernel="linear",
                        C=1.0,
                        decision_function_shape="ovo",
                        random_state=seed,
                    ),
                ),
            ],
        ),
    }


def compute_metrics(
    true_labels: np.ndarray,
    predicted_labels: np.ndarray,
) -> dict[str, float]:
    return {
        "AccuracyPercent": 100.0
        * accuracy_score(true_labels, predicted_labels),
        "MacroPrecision": precision_score(
            true_labels,
            predicted_labels,
            average="macro",
            zero_division=0,
        ),
        "MacroSensitivity": recall_score(
            true_labels,
            predicted_labels,
            average="macro",
            zero_division=0,
        ),
        "MacroF1": f1_score(
            true_labels,
            predicted_labels,
            average="macro",
            zero_division=0,
        ),
    }


def evaluate_ten_fold(
    x: np.ndarray,
    y: np.ndarray,
    group_name: str,
    config: PipelineConfig,
) -> pd.DataFrame:
    models = build_models(config.random_seed)

    cross_validation = StratifiedKFold(
        n_splits=10,
        shuffle=True,
        random_state=config.random_seed,
    )

    rows: list[dict[str, float | str]] = []

    for model_name, model in models.items():
        predictions = cross_val_predict(
            clone(model),
            x,
            y,
            cv=cross_validation,
            method="predict",
            n_jobs=None,
        )

        metrics = compute_metrics(y, predictions)

        paper_accuracy = PAPER_RESULTS[group_name][
            "ten_fold_accuracy_percent"
        ][model_name]

        rows.append({
            "Group": group_name,
            "Classifier": model_name,
            **metrics,
            "PaperAccuracyPercent": paper_accuracy,
            "AccuracyDifferenceFromPaperPercent": (
                metrics["AccuracyPercent"] - paper_accuracy
            ),
        })

    return pd.DataFrame(rows)


def evaluate_repeated_holdout(
    x: np.ndarray,
    y: np.ndarray,
    group_name: str,
    config: PipelineConfig,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    all_run_rows: list[dict[str, float | int | str]] = []

    for repetition in range(1, config.holdout_repetitions + 1):
        splitter = StratifiedShuffleSplit(
            n_splits=1,
            test_size=config.holdout_fraction,
            random_state=repetition,
        )

        train_indices, test_indices = next(splitter.split(x, y))

        x_train = x[train_indices]
        x_test = x[test_indices]
        y_train = y[train_indices]
        y_test = y[test_indices]

        models = build_models(repetition)

        for model_name, model in models.items():
            fitted_model = clone(model)
            fitted_model.fit(x_train, y_train)
            predictions = fitted_model.predict(x_test)
            metrics = compute_metrics(y_test, predictions)

            all_run_rows.append({
                "Group": group_name,
                "Repetition": repetition,
                "Classifier": model_name,
                **metrics,
            })

    all_runs = pd.DataFrame(all_run_rows)

    summary = (
        all_runs.groupby(["Group", "Classifier"], as_index=False)
        .agg(
            MeanAccuracyPercent=("AccuracyPercent", "mean"),
            StandardDeviationPercent=("AccuracyPercent", "std"),
            MedianAccuracyPercent=("AccuracyPercent", "median"),
            MinimumAccuracyPercent=("AccuracyPercent", "min"),
            MaximumAccuracyPercent=("AccuracyPercent", "max"),
            MeanMacroPrecision=("MacroPrecision", "mean"),
            MeanMacroSensitivity=("MacroSensitivity", "mean"),
            MeanMacroF1=("MacroF1", "mean"),
        )
    )

    paper_means: list[float] = []
    paper_sds: list[float] = []

    for classifier_name in summary["Classifier"]:
        paper_means.append(
            PAPER_RESULTS[group_name]["holdout_mean_percent"][
                classifier_name
            ]
        )
        paper_sds.append(
            PAPER_RESULTS[group_name]["holdout_sd_percent"][
                classifier_name
            ]
        )

    summary["PaperMeanAccuracyPercent"] = paper_means
    summary["PaperStandardDeviationPercent"] = paper_sds
    summary["MeanDifferenceFromPaperPercent"] = (
        summary["MeanAccuracyPercent"]
        - summary["PaperMeanAccuracyPercent"]
    )
    summary["StandardDeviationDifferenceFromPaperPercent"] = (
        summary["StandardDeviationPercent"]
        - summary["PaperStandardDeviationPercent"]
    )

    return summary, all_runs


def fit_and_save_final_models(
    x: np.ndarray,
    y: np.ndarray,
    group_name: str,
    config: PipelineConfig,
) -> None:
    model_directory = config.output_dir / "models"
    model_directory.mkdir(parents=True, exist_ok=True)

    for model_name, model in build_models(config.random_seed).items():
        fitted_model = clone(model)
        fitted_model.fit(x, y)

        safe_name = (
            model_name.lower()
            .replace(" ", "_")
            .replace("-", "_")
        )

        joblib.dump(
            fitted_model,
            model_directory / f"group_{group_name}_{safe_name}.joblib",
        )


def evaluate_group(
    features: pd.DataFrame,
    group_name: str,
    config: PipelineConfig,
) -> dict[str, pd.DataFrame]:
    normalized_group = group_name.strip().upper()

    group_data = features.loc[
        features["Group"] == normalized_group
    ].copy()

    if group_data.empty:
        raise ValueError(
            f"No feature rows were found for Group {normalized_group}."
        )

    x = group_data[["DominantFrequencyHz"]].to_numpy(dtype=np.float64)
    y = group_data["ClassLabel"].to_numpy(dtype=np.int64)

    ten_fold = evaluate_ten_fold(x, y, normalized_group, config)

    holdout_summary, holdout_all_runs = evaluate_repeated_holdout(
        x, y, normalized_group, config,
    )

    ten_fold.to_csv(
        config.output_dir / f"Group{normalized_group}_TenFold_Summary.csv",
        index=False,
    )
    holdout_summary.to_csv(
        config.output_dir
        / f"Group{normalized_group}_RepeatedHoldout_Summary.csv",
        index=False,
    )
    holdout_all_runs.to_csv(
        config.output_dir
        / f"Group{normalized_group}_RepeatedHoldout_AllRuns.csv",
        index=False,
    )

    fit_and_save_final_models(x, y, normalized_group, config)

    return {
        "ten_fold": ten_fold,
        "holdout_summary": holdout_summary,
        "holdout_all_runs": holdout_all_runs,
    }


## 6. Pipeline driver (`run_pipeline.py`)

This reproduces the CLI script's logic as a callable function. Originally it parsed `--dataset`, `--output`, `--group`, `--seed`, `--repetitions`, and `--save-d4` command-line arguments; here those become function parameters so it can run directly inside the notebook.

In [35]:
def run_pipeline(
    dataset: Path,
    output: Path,
    group: str = "both",
    seed: int = 2026,
    repetitions: int = 100,
    save_d4: bool = False,
) -> dict[str, dict[str, pd.DataFrame]]:
    """Reproduce the main single-channel SSVEP pipeline: full-trial db4 DWT,
    D4 reconstruction, five-second crop, FFT dominant frequency, and three
    classifiers. Equivalent to running run_pipeline.py from the command line.
    """
    groups = ("A", "B") if group == "both" else (group,)

    config = PipelineConfig(
        dataset_dir=dataset.resolve(),
        output_dir=output.resolve(),
        random_seed=seed,
        holdout_repetitions=repetitions,
        save_intermediate_d4=save_d4,
    )

    print("=" * 60)
    print("Python SSVEP main pipeline")
    print("=" * 60)
    print(f"Dataset: {config.dataset_dir}")
    print(f"Output: {config.output_dir}")
    print(f"Groups: \', \'.join(groups)")
    print("Order: full-trial DWT -> D4 reconstruction -> crop -> FFT")
    print(
        f"Stimulus window: MATLAB "
        f"{config.stimulus_start_matlab_index}:"
        f"{config.stimulus_end_matlab_index}"
    )
    print(
        f"FFT band: {config.fft_lower_hz}-"
        f"{config.fft_upper_hz} Hz"
    )
    print()

    features = extract_features(config, groups=groups)

    print()
    print("=" * 60)
    print("Feature extraction completed")
    print("=" * 60)
    print(features.groupby(["Group", "ClassLabel"]).size())
    print()

    all_results: dict[str, dict[str, pd.DataFrame]] = {}

    for group_name in groups:
        results = evaluate_group(features, group_name, config)
        all_results[group_name] = results

        print("=" * 60)
        print(f"Group {group_name}: 10-fold results")
        print("=" * 60)
        print(results["ten_fold"].to_string(index=False))
        print()

        print("=" * 60)
        print(f"Group {group_name}: repeated holdout results")
        print("=" * 60)
        print(results["holdout_summary"].to_string(index=False))
        print()

    print("=" * 60)
    print("Pipeline completed")
    print("=" * 60)
    print(f"Results were saved to: {config.output_dir}")

    return {"features": features, **all_results}


## 7. Run the pipeline

Set `dataset_dir` to the folder containing `S1.mat` ... `S35.mat` and `Freq_Phase.mat` (matches your `data/benchmark` folder), and `output_dir` to where CSVs/models should be written (matches your `results` folder).

In [36]:
DATASET_DIR = Path("../data/benchmark")
OUTPUT_DIR = Path("../results/python")

pipeline_output = run_pipeline(
    dataset=DATASET_DIR,
    output=OUTPUT_DIR,
    group="both",
    seed=2026,
    repetitions=100,
    save_d4=False,
)

features = pipeline_output["features"]
results_a = pipeline_output["A"]
results_b = pipeline_output["B"]
features.head()


Python SSVEP main pipeline
Dataset: C:\Users\Sepi\Desktop\Sepehr\UNIVERSITY\Uni Master aut\Term 2\Pattern\project\SSVEP_Article_Reproduction_404133012\data\benchmark
Output: C:\Users\Sepi\Desktop\Sepehr\UNIVERSITY\Uni Master aut\Term 2\Pattern\project\SSVEP_Article_Reproduction_404133012\results\python
Groups: ', '.join(groups)
Order: full-trial DWT -> D4 reconstruction -> crop -> FFT
Stimulus window: MATLAB 126:1375
FFT band: 7.8-15.6 Hz

Verified target-frequency mapping:
Class 1: 9.0 Hz -> dataset index 2 (MATLAB)
Class 2: 10.0 Hz -> dataset index 3 (MATLAB)
Class 3: 11.0 Hz -> dataset index 4 (MATLAB)
Class 4: 12.0 Hz -> dataset index 5 (MATLAB)

Group A: processing S10 (1/10)
Group A: processing S14 (2/10)
Group A: processing S15 (3/10)
Group A: processing S20 (4/10)
Group A: processing S22 (5/10)
Group A: processing S25 (6/10)
Group A: processing S26 (7/10)
Group A: processing S31 (8/10)
Group A: processing S32 (9/10)
Group A: processing S35 (10/10)
Group B: processing S1 (1/18)


,Group,SubjectID,ClassLabel,TargetFrequencyHz,Block,DominantFrequencyHz,DominantAmplitude
0,A,10,1,9.0,1,9.0,2.035093
1,A,10,1,9.0,2,9.0,1.464684
2,A,10,1,9.0,3,9.6,1.021598
3,A,10,1,9.0,4,8.6,1.152904
4,A,10,1,9.0,5,9.8,1.430018


### Repeated holdout summaries

In [37]:
holdout_summary_all = pd.concat(
    [results_a["holdout_summary"], results_b["holdout_summary"]],
    ignore_index=True,
)
holdout_summary_all


,Group,Classifier,MeanAccuracyPercent,StandardDeviationPercent,MedianAccuracyPercent,MinimumAccuracyPercent,MaximumAccuracyPercent,MeanMacroPrecision,MeanMacroSensitivity,MeanMacroF1,PaperMeanAccuracyPercent,PaperStandardDeviationPercent,MeanDifferenceFromPaperPercent,StandardDeviationDifferenceFromPaperPercent
0,A,Fine Tree,95.895833,2.485490,95.833333,89.583333,100.000000,0.962430,0.958958,0.958972,96.74,2.67,-0.844167,-0.184510
1,A,Linear Discriminant,96.541667,2.427758,96.875000,89.583333,100.000000,0.967585,0.965417,0.964956,97.66,2.35,-1.118333,0.077758
2,A,Linear SVM,96.291667,2.379607,95.833333,89.583333,100.000000,0.965334,0.962917,0.962404,97.66,2.35,-1.368333,0.029607
3,B,Fine Tree,80.793103,3.606402,81.034483,70.114943,89.655172,0.817056,0.807944,0.807921,90.46,2.22,-9.666897,1.386402
4,B,Linear Discriminant,82.425287,3.489804,82.758621,70.114943,90.804598,0.834775,0.824221,0.824295,91.60,0.82,-9.174713,2.669804
5,B,Linear SVM,80.804598,3.654939,81.609195,67.816092,88.505747,0.829357,0.807987,0.808470,92.68,0.67,-11.875402,2.984939
